In [ ]:
!pip install chembl-webresource-client rdkit padelpy -q

In [ ]:
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

In [ ]:
target = new_client.target
target_query = target.search('Androgen Receptor')
targets = pd.DataFrame.from_dict(target_query)

print(targets[['target_chembl_id', 'pref_name', 'organism']].head(10))

In [ ]:
activity = new_client.activity
res = activity.filter(
    target_chembl_id='CHEMBL1871',
    standard_type='IC50'
)

df = pd.DataFrame.from_dict(res)
print("Raw records:", df.shape)
print(df.columns.tolist())

In [ ]:
df2 = df[['molecule_chembl_id', 'canonical_smiles',
           'standard_value', 'standard_units']].copy()

df2 = df2.dropna(subset=['canonical_smiles', 'standard_value'])
df2['standard_value'] = pd.to_numeric(df2['standard_value'], errors='coerce')
df2 = df2.dropna(subset=['standard_value'])

df2 = df2[df2['standard_units'] == 'nM']

def classify(value):
    if value <= 1000:
        return 'active'
    elif value >= 10000:
        return 'inactive'
    else:
        return 'intermediate'

df2['bioactivity_class'] = df2['standard_value'].apply(classify)
df2 = df2[df2['bioactivity_class'] != 'intermediate']

df2 = df2.drop_duplicates(subset='canonical_smiles')
df2 = df2.reset_index(drop=True)

print("Final dataset shape:", df2.shape)
print(df2['bioactivity_class'].value_counts())

In [ ]:
df2['standard_value'] = df2['standard_value'].clip(lower=1e-9)
df2['pIC50'] = -np.log10(df2['standard_value'] * 1e-9)

print("pIC50 stats:")
print(df2['pIC50'].describe())

In [ ]:
from rdkit.Chem import Descriptors

def compute_lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [None]*4
    return [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol)
    ]

lipinski = df2['canonical_smiles'].apply(
    lambda x: pd.Series(
        compute_lipinski(x),
        index=['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']
    )
)

df_lipinski = pd.concat([df2, lipinski], axis=1)
df_lipinski = df_lipinski.dropna(subset=['MW'])
df_lipinski = df_lipinski.reset_index(drop=True)

print("Shape after Lipinski:", df_lipinski.shape)
print(df_lipinski[['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']].describe())

In [ ]:
def compute_maccs(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = MACCSkeys.GenMACCSKeys(mol)
    return list(fp.ToBitString())

maccs_list = df_lipinski['canonical_smiles'].apply(compute_maccs)
maccs_list = maccs_list.dropna()

df_maccs = pd.DataFrame(
    maccs_list.tolist(),
    index=maccs_list.index,
    columns=[f'MACCS_{i}' for i in range(167)]
)

df_lipinski = df_lipinski.loc[df_maccs.index].reset_index(drop=True)
df_maccs = df_maccs.reset_index(drop=True)

df_maccs = df_maccs.apply(pd.to_numeric)

print("MACCS shape:", df_maccs.shape)
print("Lipinski shape:", df_lipinski.shape)

In [ ]:
X = pd.concat([
    df_lipinski[['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']],
    df_maccs
], axis=1)

y = (df_lipinski['bioactivity_class'] == 'active').astype(int)

print("Feature matrix shape:", X.shape)
print("Label distribution:")
print(y.value_counts())

df_lipinski.to_csv('df_clean.csv', index=False)
X.to_csv('features.csv', index=False)
y.to_csv('labels.csv', index=False)

print("\nFiles saved successfully")

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                              roc_curve, confusion_matrix)
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler

selector = VarianceThreshold(threshold=0.01)
X_filtered = selector.fit_transform(X)

print(f"Features before filtering: {X.shape[1]}")
print(f"Features after filtering: {X_filtered.shape[1]}")

X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(pd.Series(y_train).value_counts())

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:,1]
rf_auc = roc_auc_score(y_test, rf_proba)

print("=== Random Forest Results ===")
print(f"AUC: {rf_auc:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, rf_preds,
      target_names=['Inactive', 'Active']))

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)
lr.fit(X_train_scaled, y_train)

lr_preds = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:,1]
lr_auc = roc_auc_score(y_test, lr_proba)

print("=== Logistic Regression Results ===")
print(f"AUC: {lr_auc:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, lr_preds,
      target_names=['Inactive', 'Active']))

In [14]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(8, 6))

fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
plt.plot(fpr_rf, tpr_rf,
         label=f'Random Forest (AUC = {rf_auc:.3f})',
         color='steelblue', linewidth=2)

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
plt.plot(fpr_lr, tpr_lr,
         label=f'Logistic Regression (AUC = {lr_auc:.3f})',
         color='orange', linewidth=2)

plt.plot([0,1], [0,1], 'k--', label='Random Baseline', linewidth=1)

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve: AR Inhibitor Bioactivity Prediction', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: roc_curve.png")

In [ ]:
feature_names = list(X.columns[selector.get_support()])

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:20]

plt.figure(figsize=(12, 6))
plt.bar(range(20), importances[indices], color='steelblue')
plt.xticks(range(20),
           [feature_names[i] for i in indices],
           rotation=45, ha='right', fontsize=9)
plt.xlabel('Molecular Descriptor')
plt.ylabel('Importance Score')
plt.title('Top 20 Descriptors for AR Inhibitor Activity Prediction')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop 10 most important descriptors:")
for i in range(10):
    print(f"{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}")

In [ ]:
cv_scores = cross_val_score(
    rf, X_filtered, y,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

print("5-Fold Cross-Validation AUC:")
print(f"Mean: {cv_scores.mean():.3f}")
print(f"Std:  {cv_scores.std():.3f}")
print(f"All folds: {cv_scores.round(3)}")

In [ ]:
train_proba = rf.predict_proba(X_train)[:,1]
train_auc = roc_auc_score(y_train, train_proba)

print(f"Training AUC:           {train_auc:.3f}")
print(f"Test AUC:               {rf_auc:.3f}")
print(f"Cross-validation AUC:   {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"\nGap (train - CV):       {train_auc - cv_scores.mean():.3f}")

print(f"\nDataset size: {X_filtered.shape[0]} compounds")
print(f"Training size per fold (4/5): {int(X_filtered.shape[0]*0.8)}")
print(f"Test size per fold (1/5): {int(X_filtered.shape[0]*0.2)}")

In [ ]:
rf_tuned = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,          # limit tree depth
    min_samples_leaf=5,    # require min samples at leaves
    min_samples_split=10,  # require min samples to split
    max_features='sqrt',   # limit features per split
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train, y_train)

tuned_preds = rf_tuned.predict(X_test)
tuned_proba = rf_tuned.predict_proba(X_test)[:,1]
tuned_auc = roc_auc_score(y_test, tuned_proba)

cv_tuned = cross_val_score(
    rf_tuned, X_filtered, y,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

train_proba_tuned = rf_tuned.predict_proba(X_train)[:,1]
train_auc_tuned = roc_auc_score(y_train, train_proba_tuned)

print("=== Tuned Random Forest Results ===")
print(f"Training AUC:         {train_auc_tuned:.3f}")
print(f"Test AUC:             {tuned_auc:.3f}")
print(f"Cross-validation AUC: {cv_tuned.mean():.3f} ± {cv_tuned.std():.3f}")
print(f"All CV folds:         {cv_tuned.round(3)}")
print(f"\nGap (train - CV):     {train_auc_tuned - cv_tuned.mean():.3f}")
print("\nClassification Report:")
print(classification_report(y_test, tuned_preds,
      target_names=['Inactive', 'Active']))

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Per-fold class distribution:")
print("-" * 50)
for fold, (train_idx, val_idx) in enumerate(skf.split(X_filtered, y)):
    y_val_fold = y.iloc[val_idx]
    active = y_val_fold.sum()
    inactive = (1 - y_val_fold).sum()

    rf_fold = RandomForestClassifier(
        n_estimators=200, max_depth=10,
        min_samples_leaf=5, min_samples_split=10,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    rf_fold.fit(X_filtered[train_idx], y.iloc[train_idx])
    fold_proba = rf_fold.predict_proba(X_filtered[val_idx])[:,1]
    fold_auc = roc_auc_score(y_val_fold, fold_proba)

    print(f"Fold {fold+1}: Active={active}, Inactive={inactive}, "
          f"Ratio={active/inactive:.2f}, AUC={fold_auc:.3f}")

In [ ]:
stratified_aucs = [0.915, 0.921, 0.922, 0.901, 0.928]
mean_auc = np.mean(stratified_aucs)
std_auc = np.std(stratified_aucs)

print("=" * 50)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 50)
print(f"\nModel: Random Forest (n=200, max_depth=10)")
print(f"Dataset: {X_filtered.shape[0]} AR inhibitors (ChEMBL1871)")
print(f"Features: {X_filtered.shape[1]} (Lipinski + MACCS)")
print(f"\nTest AUC:                    {tuned_auc:.3f}")
print(f"Stratified 5-Fold CV AUC:    {mean_auc:.3f} ± {std_auc:.3f}")
print(f"Test Accuracy:               88%")
print(f"Active Recall:               90%")
print(f"\nTop predictive descriptors:")
print(f"1. MW (Molecular Weight)")
print(f"2. LogP (Lipophilicity)")
print(f"3. MACCS_41 (Ring system)")
print(f"4. MACCS_66 (Structural pattern)")
print(f"5. NumHAcceptors")
print(f"\nKey finding: Non-linear RF outperforms linear LR")
print(f"(AUC 0.919 vs 0.868), confirming complex descriptor")
print(f"relationships in AR inhibitor activity.")